# 02 — Shared BPE tokenization

> **Status:** implementation in progress.

- **Mapped issue:** [#3](https://github.com/majorgilles/transformer-2017-reproduction/issues/3)
- **Depends on:** `01_data_contracts_provenance.ipynb` / issue #2.


In [1]:
#| default_exp tokenization


## Stable special-token contract

The tokenizer reserves padding, unknown, beginning-of-sequence, and end-of-sequence symbols before learning ordinary BPE pieces. Their tuple order fixes IDs 0–3, so later datasets, embeddings, checkpoints, and inference code interpret the same integers consistently.


In [2]:
#| export
from __future__ import annotations

from collections.abc import Iterable
from enum import StrEnum
from typing import Final, Self

from pydantic import BaseModel, ConfigDict, Field, model_validator
from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers


class SpecialToken(StrEnum):
    """Reserved tokens with fixed vocabulary positions."""

    PAD = "<pad>"  # Fills unused positions when sequences are batched to one length.
    UNK = "<unk>"  # Represents input text that is absent from the learned vocabulary.
    BOS = "<bos>"  # Marks the beginning of a token sequence.
    EOS = "<eos>"  # Marks the end of a token sequence.


SPECIAL_TOKENS: Final[tuple[str, ...]] = tuple(token.value for token in SpecialToken)

In [3]:
assert SPECIAL_TOKENS == ("<pad>", "<unk>", "<bos>", "<eos>")
assert len(SPECIAL_TOKENS) == len(set(SPECIAL_TOKENS))

## Validated training configuration

A tokenizer artifact is meaningful only when its training settings are known. This frozen Pydantic model rejects unknown fields, impossible vocabulary sizes, and reordered special tokens so invalid settings fail before training begins.


In [4]:
#| export
class BPETrainingConfig(BaseModel):
    """Validated settings that determine a learned BPE vocabulary."""

    model_config = ConfigDict(frozen=True, extra="forbid", strict=True)

    vocab_size: int = Field(gt=len(SPECIAL_TOKENS))  # Maximum learned vocabulary size.
    min_frequency: int = Field(ge=1)  # Minimum corpus count required for a merge.
    special_tokens: tuple[str, ...] = SPECIAL_TOKENS  # Fixed values and ID order.

    @model_validator(mode="after")
    def require_canonical_special_tokens(self) -> Self:
        if self.special_tokens != SPECIAL_TOKENS:
            raise ValueError("special tokens must use the canonical values and order")
        return self

In [5]:
# Small test configuration; this is not the canonical WMT vocabulary size.
fixture_config = BPETrainingConfig(vocab_size=46, min_frequency=2)

assert fixture_config.special_tokens == SPECIAL_TOKENS
assert fixture_config.model_dump() == {
    "vocab_size": 46,
    "min_frequency": 2,
    "special_tokens": ("<pad>", "<unk>", "<bos>", "<eos>"),
}

## Shared BPE training pipeline

Both English and German strings flow through one trainer, producing one shared token-to-ID space. Before BPE sees the text, `Metaspace` replaces whitespace with one visible marker character. `replacement` selects that marker, `prepend_scheme="always"` also places it before the first word, and `split=True` tells the pre-tokenizer to split at marker positions. The matching Metaspace decoder uses the same marker and prepend scheme to restore readable spaces.


In [6]:
#| export
def train_bpe(texts: Iterable[str], config: BPETrainingConfig) -> Tokenizer:
    """Train a shared BPE tokenizer from an iterable of text."""

    # The model learns merge rules and falls back to <unk> for unseen symbols.
    tokenizer = Tokenizer(models.BPE(unk_token=SpecialToken.UNK.value))
    # Metaspace makes whitespace visible to BPE instead of discarding word boundaries.
    tokenizer.pre_tokenizer = pre_tokenizers.Metaspace(
        # Replace each whitespace boundary with the conventional ▁ marker.
        replacement="▁",
        # Add the marker before the first word as well as words following spaces.
        prepend_scheme="always",
        # Split at marker positions so boundaries become explicit pre-tokens.
        split=True,
    )

    # The trainer inserts special tokens first, preserving their canonical IDs.
    trainer = trainers.BpeTrainer(
        vocab_size=config.vocab_size,
        min_frequency=config.min_frequency,
        special_tokens=list(config.special_tokens),
        show_progress=False,
    )
    tokenizer.train_from_iterator(texts, trainer=trainer)

    # The decoder reverses the Metaspace transformation after token IDs are decoded.
    tokenizer.decoder = decoders.Metaspace(
        # Interpret the same ▁ marker as whitespace; this must match the pre-tokenizer.
        replacement="▁",
        # Match the synthetic marker placed before the first word.
        prepend_scheme="always",
    )
    return tokenizer

## Deterministic bilingual fixture

This bounded English-German corpus contains related word forms so the learned vocabulary exhibits a realistic mixture of whole frequent words and reusable subword pieces. A vocabulary limit of 46 and minimum frequency of 2 are fixture-only settings chosen to make those merges visible; they are not the canonical WMT choices.


In [7]:
fixture_texts = (
    "the cat sleeps",
    "the cats sleep",
    "the cat slept",
    "a sleeping cat",
    "die katze schläft",
    "die katzen schlafen",
    "die katze schlief",
    "eine schlafende katze",
)

fixture_tokenizer = train_bpe(fixture_texts, fixture_config)

for expected_id, token in enumerate(SPECIAL_TOKENS):
    assert fixture_tokenizer.token_to_id(token) == expected_id

## Visible result: one vocabulary for both languages

The result below encodes both fixture sentences and new combinations with the same vocabulary. Frequent forms can become whole tokens, while related forms split into shared stems and endings such as `▁cat` + `s` or `▁schlaf` + `en`. The `▁` character is the conventional visible space marker used internally by Metaspace.


In [8]:
demonstration_texts = (
    "the cats sleep",
    "die katzen schlafen",
    "the cat is sleeping",
    "die katze kann schlafen",
)

print(f"Learned shared vocabulary size: {fixture_tokenizer.get_vocab_size()}")
print("Boundary marker: ▁ means a space or the start of a sentence.")
for text in demonstration_texts:
    encoded = fixture_tokenizer.encode(text)
    decoded = fixture_tokenizer.decode(encoded.ids)
    print(f"{text!r} -> {encoded.tokens} -> {encoded.ids} -> {decoded!r}")

Learned shared vocabulary size: 46
Boundary marker: ▁ means a space or the start of a sentence.
'the cats sleep' -> ['▁the', '▁cat', 's', '▁sleep'] -> [40, 29, 16, 41] -> 'the cats sleep'
'die katzen schlafen' -> ['▁die', '▁katze', 'n', '▁schlaf', 'en'] -> [39, 33, 14, 45, 43] -> 'die katzen schlafen'
'the cat is sleeping' -> ['▁the', '▁cat', '▁', 'i', 's', '▁sleep', 'in', 'g'] -> [40, 29, 20, 11, 16, 41, 44, 9] -> 'the cat is sleeping'
'die katze kann schlafen' -> ['▁die', '▁katze', '▁', 'k', 'a', 'n', 'n', '▁schlaf', 'en'] -> [39, 33, 20, 12, 4, 14, 14, 45, 43] -> 'die katze kann schlafen'


## Visible round-trip evidence

Each example is encoded into shared-vocabulary pieces and decoded back into text.
The printed rows make the behavior inspectable, while the checks fail execution if
text changes or the tokenizer unexpectedly uses `<unk>`.

In [10]:
round_trip_texts = fixture_texts + demonstration_texts

print("Round-trip evidence:")
for text in round_trip_texts:
    encoded = fixture_tokenizer.encode(text)
    decoded = fixture_tokenizer.decode(encoded.ids)
    used_unknown = SpecialToken.UNK.value in encoded.tokens

    print(
        f"input={text!r}\n"
        f"  tokens={encoded.tokens}\n"
        f"  decoded={decoded!r}\n"
        f"  exact_match={decoded == text}, used_unknown={used_unknown}"
    )

    assert decoded == text
    assert not used_unknown

Round-trip evidence:
input='the cat sleeps'
  tokens=['▁the', '▁cat', '▁sleep', 's']
  decoded='the cat sleeps'
  exact_match=True, used_unknown=False
input='the cats sleep'
  tokens=['▁the', '▁cat', 's', '▁sleep']
  decoded='the cats sleep'
  exact_match=True, used_unknown=False
input='the cat slept'
  tokens=['▁the', '▁cat', '▁sl', 'ep', 't']
  decoded='the cat slept'
  exact_match=True, used_unknown=False
input='a sleeping cat'
  tokens=['▁', 'a', '▁sleep', 'in', 'g', '▁cat']
  decoded='a sleeping cat'
  exact_match=True, used_unknown=False
input='die katze schläft'
  tokens=['▁die', '▁katze', '▁schl', 'ä', 'f', 't']
  decoded='die katze schläft'
  exact_match=True, used_unknown=False
input='die katzen schlafen'
  tokens=['▁die', '▁katze', 'n', '▁schlaf', 'en']
  decoded='die katzen schlafen'
  exact_match=True, used_unknown=False
input='die katze schlief'
  tokens=['▁die', '▁katze', '▁schl', 'ie', 'f']
  decoded='die katze schlief'
  exact_match=True, used_unknown=False
input='eine

## Goal

Learn a shared English-German BPE vocabulary and give it a stable identity.


## Paper and project contract

*Attention Is All You Need* reports a shared English-German BPE vocabulary of about 37,000 tokens and later shares the source embedding, target embedding, and pre-softmax weight matrix. This project keeps one shared vocabulary but may scale its size after measuring vocabulary use and sequence-length inflation. Training text must come only from the approved training split exposed by `iter_bpe_training_text`; development and final-test text must not influence the learned vocabulary.

The tokenizer identity will ultimately bind the training configuration, ordered vocabulary and merges, tokenizer-library version, and source dataset-manifest identity.


## Required deliverables

- Typed tokenizer API
- Fixture-trained BPE artifact
- Round-trip tests
- Vocabulary and sequence-length report


## Planned implementation sections

1. Paper and contract references
2. Typed implementation
3. Focused tests
4. Deterministic visible result
5. Exported API and artifact identities


## Explicitly deferred

Embeddings, positions, attention, and neural model code.


## HITL checkpoint

Approve vocabulary evidence, special tokens, behavior, and artifact identity.
